# Lab 2 · One defensible feature

Invented teaching dataset `w04_race_weekends_synthetic_v3.csv` (18 weekends × 20 drivers). Target: `top10`. Metric: validation balanced accuracy. Split by week: train 1–10, validation 11–14, test 15–18 (sealed: never fitted, transformed or scored in this lab).

**Members:** Francisco Polo, Milan Kurte

### 1. Data and week-based split

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score

path = Path("w04_race_weekends_synthetic_v3.csv")
if not path.exists():
    raise FileNotFoundError("Open the notebook in the folder containing its CSV file.")
data = pd.read_csv(path)
print("Rows, columns:", data.shape)
print("Columns:", data.columns.tolist())

Rows, columns: (360, 9)
Columns: ['week', 'driver_id', 'qualifying_position', 'practice_gap_s', 'previous_top10_rate', 'grid_penalty', 'circuit_type', 'rain_forecast', 'top10']


In [ ]:
train = data.loc[data.week <= 10].copy()
valid = data.loc[data.week.between(11, 14)].copy()
n_test_sealed = int(data.week.between(15, 18).sum())  # Only counted; weeks 15–18 stay sealed.

base_numeric = ["qualifying_position", "practice_gap_s", "previous_top10_rate", "grid_penalty"]
categorical = ["circuit_type", "rain_forecast"]
print("Train / validation / sealed test rows:", len(train), len(valid), n_test_sealed)
print("Train weeks:", sorted(train.week.unique()))
print("Validation weeks:", sorted(valid.week.unique()))
assert train.week.max() == 10 and valid.week.min() == 11 and valid.week.max() == 14
assert set(train.index).isdisjoint(valid.index)

Train / validation / sealed test rows: 200 80 80
Train weeks: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Validation weeks: [11, 12, 13, 14]


### 2. Proposal (Max 120 words)

***Feature name***: estimated_grid_position

***Formula***: qualifying_position + grid_penalty

***Source columns***: qualifying_position and grid_penalty

***When known***: Both values are known after qualifying and penalty applications, prior to the race start.

***Expected direction of change***: I expect the validation balanced accuracy to increase. Knowing the estimated starting grid position provides a more realistic representation of a driver's initial race advantage than the raw qualifying position alone, as grid penalties significantly alter the starting order.

### 3. Candidate feature

`estimated_grid_position = qualifying_position + grid_penalty`. Both inputs are known after qualifying, before the race. Computed on train and validation rows only.

In [ ]:
NEW_FEATURE = "estimated_grid_position"

def add_candidate(df):
    out = df.copy()
    out[NEW_FEATURE] = out["qualifying_position"] + out["grid_penalty"]
    return out

train = add_candidate(train)
valid = add_candidate(valid)

features_without = base_numeric + categorical
features_with = base_numeric + [NEW_FEATURE] + categorical
forbidden = {"top10", "week", "driver_id"}
assert set(features_with) - set(features_without) == {NEW_FEATURE}
assert not forbidden & set(features_with)
y_train, y_valid = train.top10, valid.top10
print(train[["qualifying_position", "grid_penalty", NEW_FEATURE]].head(5).to_string(index=False))

 qualifying_position  grid_penalty  estimated_grid_position
                   4             0                        4
                   3             0                        3
                  13             0                       13
                   6             0                        6
                   1             5                        6


### 4. Comparison on validation (weeks 11–14)

Both pipelines come from the same builder; the only difference is the numeric column list. All `fit` calls use train rows (weeks 1–10).

In [ ]:
def make_pipeline(numeric_cols):
    numeric_pipe = Pipeline([
        ("fill", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("fill", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocess = ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical),
    ])
    return Pipeline([
        ("prep", preprocess),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ])

baseline = DummyClassifier(strategy="most_frequent").fit(train[features_without], y_train)
pipe_without = make_pipeline(base_numeric).fit(train[features_without], y_train)
pipe_with = make_pipeline(base_numeric + [NEW_FEATURE]).fit(train[features_with], y_train)

In [ ]:
results = pd.DataFrame([
    ("Dummy baseline (most frequent)", 0, balanced_accuracy_score(y_valid, baseline.predict(valid[features_without]))),
    ("Pipeline without feature", len(features_without), balanced_accuracy_score(y_valid, pipe_without.predict(valid[features_without]))),
    ("Pipeline with feature", len(features_with), balanced_accuracy_score(y_valid, pipe_with.predict(valid[features_with]))),
], columns=["model", "n_input_features", "validation_balanced_accuracy"])
results["validation_balanced_accuracy"] = results["validation_balanced_accuracy"].round(3)
print(results.to_string(index=False))
assert len(results) == 3

                         model  n_input_features  validation_balanced_accuracy
Dummy baseline (most frequent)                 0                         0.500
      Pipeline without feature                 6                         0.862
         Pipeline with feature                 7                         0.862


### 5. Decision (Max 120 words)

***Decision***: Reject the feature.

***Scores***: Dummy baseline = 0.500 | Without feature = 0.862 | With feature = 0.862.
***Reason***: The balanced accuracy remained identical. This occurs because the new feature is an exact linear combination of two existing columns. Since Logistic Regression is a linear model, it already optimally weights qualifying_position and grid_penalty independently. Adding their direct sum provides no new mathematical variance or non-linear information for the model to learn from.

***Limitation***: Real F1 performance cannot be inferred from this synthetic dataset. Additionally, linear models struggle to extract value from linear feature combinations unless we capture more complex, non-linear interactions.

### 6. Brief AI recommendation audit & AI-use record

***Reject the advice.*** Methodological reason: Fitting an imputer on all 18 weeks causes Data Leakage. Even without the target variable, the imputer learns the statistical distribution (e.g., median/mode) of the validation and future test sets, violating the strictly historical nature of prediction.
Evidence: In my notebook, the data is split first (Section 1), and the pipeline .fit() is strictly applied only to the train set (weeks 1-10) in Section 4.


AI-use record:

* Prompt: Instructed Claude to split data cleanly by weeks, create the estimated_grid_position feature, and build two identical pipelines to compare them, explicitly forbidding any interaction with test weeks (15-18).

* Suggestion: Claude generated the strict week-based split logic and the make_pipeline() implementation.

* Accepted/Rejected: Accepted the code structure to avoid data leakage; rejected AI generation of the analytical text and conclusions.

* Verification: Manually verified the shape of the split datasets (200 train rows, 80 validation rows) and confirmed that test weeks were never materialized in the code.

* Limitation: AI coding tools abstract away statistical logic; if prompted loosely, they can easily introduce data leakage without throwing any errors.




### Environment

In [ ]:
import platform
print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.14
pandas: 2.3.1
numpy: 1.26.4
scikit-learn: 1.9.1
